# 第10周 · 课堂练习：卷积的算术

四组练习，全部在 CPU 上几秒钟跑完：

| 练习 | 内容 |
| --- | --- |
| E1 | 输出尺寸公式 |
| E2 | 手写 `conv2d`，与 PyTorch 对照（顺便弄清"卷积"与"互相关"） |
| E3 | 数一个小 CNN 的参数 |
| E4 | 修复 Embedding → Conv1d 的转置 bug |

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from zero2researcher.checks import week10 as ck

## E1 · 输出尺寸公式

$$W_{\text{out}} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

把它写成函数 `conv_output_size(W, K, P=0, S=1)`，返回**整数**。写之前先在纸上算三题，运行后对答案：

1. 32×32 的 CIFAR 图，5×5 的核，不补边，步长 1 → ?
2. 28×28 的 MNIST 图，3×3 的核，`padding=1`，步长 2 → ?
3. 长度 256 的影评，宽度 7 的 `Conv1d` → ?（一维卷积也是同一个公式）

In [ ]:
def conv_output_size(W, K, P=0, S=1):
    """输入边长 W、核大小 K、补边 P、步长 S → 输出边长（整数）"""
    # 在这里写你的代码
    raise NotImplementedError


print(conv_output_size(32, 5), conv_output_size(28, 3, P=1, S=2), conv_output_size(256, 7))
ck.conv_output_size(conv_output_size)

第 2 题最容易错：$(28 - 3 + 2)/2 = 13.5$，向下取整再加 1 得 14。用 PyTorch 亲眼确认一次：

In [ ]:
x = torch.zeros(1, 1, 28, 28)
print("nn.Conv2d(K=3, P=1, S=2) 的输出:", tuple(nn.Conv2d(1, 1, 3, padding=1, stride=2)(x).shape[-2:]))

## E2 · 手写 `conv2d`，与 PyTorch 对照

**不看课件**，写出 `conv2d_numpy(img, kernel)`：valid（不补边）、步长 1，输入是两个二维 NumPy 数组，返回二维 NumPy 数组。

写完与 `torch.nn.functional.conv2d` 对照。它要求四维输入 `(N, C, H, W)` 和四维的核 `(C_out, C_in, kh, kw)`——用 `[None, None]` 在最前面补出两个长度为 1 的维度，算完再用 `[0, 0]` 取回来。

In [ ]:
def conv2d_numpy(img, kernel):
    """valid、步长 1 的二维卷积（PyTorch 口径）"""
    # 在这里写你的代码
    raise NotImplementedError


rng = np.random.default_rng(0)
img = rng.normal(size=(6, 7))
kernel = np.array([[1.0, 2.0], [0.0, -1.0]])        # 故意选一个不对称的核
mine = conv2d_numpy(img, kernel)
theirs = F.conv2d(torch.tensor(img)[None, None], torch.tensor(kernel)[None, None])[0, 0].numpy()
print("输出 shape:", mine.shape, "  与 F.conv2d 的最大差异:", np.abs(mine - theirs).max())
ck.conv2d_numpy(conv2d_numpy)

### 冷知识（但会坑人）：PyTorch 的"卷积"其实是互相关

数学和信号处理教科书里的**卷积**，要先把核**旋转 180°** 再滑动；不翻转、直接滑动的那种叫**互相关**（cross-correlation）。深度学习框架实现的全是互相关，只是沿用了"卷积"这个名字——[nn.Conv2d 的文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)公式里写的就是互相关运算符 $\star$。

对**学出来的**核，这无所谓（翻不翻转，模型都会学到一个对应的版本）；但照搬**手工设计**的核（Sobel、高斯模糊……）时，方向和正负号可能反过来。用 SciPy 的教科书卷积验证：

In [ ]:
from scipy.signal import convolve2d

true_conv = convolve2d(img, kernel, mode="valid")                 # 教科书卷积：先翻转核
print("教科书卷积 vs 我们的（不翻转核）:", np.abs(true_conv - mine).max())
print("教科书卷积 vs 我们的（手动把核翻转 180°）:", np.abs(true_conv - conv2d_numpy(img, kernel[::-1, ::-1])).max())

第二行是 0：**教科书卷积 = 把核翻转 180° 之后的互相关**。对称的核（比如 3×3 全 1 的模糊核）翻不翻转都一样；Sobel 的 $G_x$ 翻转 180° 后正负号恰好颠倒——所以同一个 $G_x$ 矩阵，按教科书卷积作用（当前版本的 Wikipedia 就把公式里的 $*$ 写成卷积）和按 PyTorch 的互相关作用，得到的边缘图差一个符号。照搬手工设计的核时，先确认资料用的是卷积还是互相关。

## E3 · 数参数

下面这个 MNIST 小 CNN 一共有多少个可训练参数？**先手算**（写成算式），再用代码验证。

```python
nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 5 * 5, 10),
)
```

提示：

- `Conv2d(C_in, C_out, k)` 的参数 = $C_{out} \times C_{in} \times k \times k$（权重）$+\ C_{out}$（每个输出通道一个偏置）；
- `Linear(d_in, d_out)` 的参数 = $d_{in} \times d_{out} + d_{out}$；
- ReLU、MaxPool、Flatten 没有参数；
- 全连接层的输入为什么是 $16 \times 5 \times 5$？用 E1 的公式追踪边长：28 → ? → ? → ? → 5。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print("手算:", n_params)
ck.cnn_param_count(n_params)

In [ ]:
net = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 5 * 5, 10),
)
print("代码验证:", sum(p.numel() for p in net.parameters()))
x = torch.zeros(1, 1, 28, 28)
for layer in net:
    x = layer(x)
    print(f"{layer.__class__.__name__:9s} → {tuple(x.shape)}")

对比：同样输入的 MLP 784 → 256 → 10 有 203,530 个参数，是它的近 40 倍。

## E4 · 修复 Embedding → Conv1d 的转置 bug

同学交来一个 TextCNN，一运行就报错。先读报错：

In [ ]:
class BuggyTextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, n_filters=16, kernel_sizes=(3, 4, 5), n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters * len(kernel_sizes), n_classes)

    def forward(self, x):
        e = self.embedding(x)
        pooled = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        return self.fc(torch.cat(pooled, dim=1))


torch.manual_seed(0)
xb = torch.randint(2, 1000, (8, 256))       # 假装是一个 batch：8 条影评、每条 256 个词 id

In [ ]:
BuggyTextCNN(1000)(xb)

报错说 Conv1d 期待 32 个通道，却收到了 256 个——它把**长度维**当成了通道维。

**第一步**：写一个函数 `to_conv_layout(e)`，把 `(N, L, C)` 变成 `(N, C, L)`。很多人的第一反应是 `e.reshape(N, C, L)`——先看看它为什么是错的：

In [ ]:
e = torch.arange(6.0).reshape(1, 3, 2)     # N=1、L=3 个词、C=2 维：三个词向量 [0,1] [2,3] [4,5]
print("原始 (N, L, C)：每行一个词\n", e[0])
print("reshape 成 (N, C, L)：\n", e.reshape(1, 2, 3)[0], "\n  ← 形状对了，但 [0, 1, 2] 不是任何一个通道")
print("transpose(1, 2)：\n", e.transpose(1, 2)[0], "\n  ← 第 0 行 = 每个词的第 0 维 [0, 2, 4]")

In [ ]:
def to_conv_layout(e):
    """(N, L, C) → (N, C, L)"""
    # 在这里写你的代码
    raise NotImplementedError


ck.conv_layout(to_conv_layout)

**第二步**：写出修好的 `TextCNN`——结构与 `BuggyTextCNN` 完全相同，只在 `forward` 里用上 `to_conv_layout`。8 条长度 256 的影评应输出 `(8, 2)` 的 logits。自查还会用长度 100 的输入再测一次：全局 max pooling 与序列长度无关，任何长度都应输出 `(N, 2)`。

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, n_filters=16, kernel_sizes=(3, 4, 5), n_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters * len(kernel_sizes), n_classes)

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError


print(TextCNN(1000)(xb).shape)
ck.textcnn_shapes(TextCNN)

**最后看一个更阴险的版本**：如果 embedding 维度**恰好等于**序列长度，不转置也不会报错——

In [ ]:
silent = BuggyTextCNN(1000, emb_dim=256)        # emb_dim = 256 = 序列长度
print("有 bug 的模型照样输出:", tuple(silent(xb).shape))

它会正常训练、损失也会下降，只是卷积在"词向量的 256 个维度"上滑动，而不是沿着词的顺序滑动——学到的东西毫无意义。**形状对了不等于对了**：这就是为什么每一行都要注释 shape，并且用一个"故意不对称"的小例子（比如上面的 `e`）检查数值。